# Lab 06 · Anomaly Detection & Time-Series Mining
**Advanced Data Mining — M.Sc. Data Science** · Dr. Ali Louati · Prince Sattam Bin Abdulaziz University
Companion to **Chapter 6**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/advanced-data-mining/blob/main/labs/Lab06_Anomaly_Detection_and_Time_Series.ipynb)

**In this lab you will:**
- detect outliers with the IQR rule, Isolation Forest and LOF
- decompose a time series into trend, seasonality and residual
- forecast with lag features and validate with TimeSeriesSplit
- beat (or not!) the seasonal-naïve baseline

> **How to use:** run the cells from top to bottom (`Shift + Enter`). Cells marked ✏️ are for you to modify.
> Every step works with the built-in data set; set `USE_MY_CSV = True` to try your own CSV file.

In [ ]:
# Setup: libraries used in this lab (all pre-installed on Google Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (8, 4.5)
np.random.seed(42)

## Part A · Anomaly detection
### 1 · A data set with planted anomalies

In [ ]:
from sklearn.datasets import make_blobs
X_normal, _ = make_blobs(n_samples=400, centers=[[0, 0], [4, 4]], cluster_std=[0.8, 0.5], random_state=1)
X_anom = np.random.uniform(-4, 8, size=(20, 2))
X = np.vstack([X_normal, X_anom])
is_anomaly = np.r_[np.zeros(400), np.ones(20)]
plt.scatter(X[:, 0], X[:, 1], c=is_anomaly, cmap="coolwarm", s=15)
plt.title("Red = planted anomalies"); plt.show()

### 2 · Isolation Forest and Local Outlier Factor

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score

iso = IsolationForest(n_estimators=300, contamination=0.05, random_state=0).fit(X)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)

pred_iso = iso.predict(X)                # -1 = anomaly
pred_lof = lof.fit_predict(X)
score_iso = -iso.score_samples(X)        # higher = more anomalous
score_lof = -lof.negative_outlier_factor_

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, name in [(axes[0], pred_iso, "Isolation Forest"), (axes[1], pred_lof, "LOF")]:
    ax.scatter(X[:, 0], X[:, 1], c=(pred == -1), cmap="coolwarm", s=15); ax.set_title(name + " (red = flagged)")
plt.show()
print("PR-AUC Isolation Forest:", round(average_precision_score(is_anomaly, score_iso), 3))
print("PR-AUC LOF             :", round(average_precision_score(is_anomaly, score_lof), 3))

## Part B · Time-series mining
### 3 · Monthly airline passengers (1949–1960)

In [ ]:
flights = sns.load_dataset("flights")
flights["date"] = pd.to_datetime(flights["year"].astype(str) + "-" + flights["month"].astype(str), format="%Y-%b")
ts = flights.set_index("date")["passengers"]
ts.plot(title="Monthly airline passengers"); plt.ylabel("passengers (thousands)"); plt.show()

### 4 · Decomposition: trend + seasonality + residual (multiplicative → use log)

In [ ]:
log_ts = np.log(ts)
trend = log_ts.rolling(12, center=True).mean()                    # 12-month moving average
detrended = log_ts - trend
seasonal = detrended.groupby(detrended.index.month).transform("mean")
residual = log_ts - trend - seasonal

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, s, name in zip(axes, [log_ts, trend, seasonal, residual], ["log(observed)", "trend", "seasonal", "residual"]):
    s.plot(ax=ax); ax.set_ylabel(name)
plt.tight_layout(); plt.show()

In [ ]:
# Anomalies in a time series = unusually large residuals (|z| > 2.5)
z = (residual - residual.mean()) / residual.std()
odd = z[abs(z) > 2.5]
ax = ts.plot(); ax.scatter(odd.index, ts[odd.index], color="red", zorder=3, label="unusual month")
ax.legend(); ax.set_title("Residual-based anomaly detection"); plt.show()

### 5 · Forecasting with lag features + rolling-origin validation

In [ ]:
data = pd.DataFrame({"y": log_ts})
for lag in [1, 2, 3, 12]:
    data[f"lag_{lag}"] = data["y"].shift(lag)
data["month"] = data.index.month
data = data.dropna()
X_ts, y_ts = data.drop(columns="y"), data["y"]
X_ts.head()

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

tscv = TimeSeriesSplit(n_splits=4, test_size=12)
rows = []
for fold, (tr, te) in enumerate(tscv.split(X_ts)):
    model = LinearRegression().fit(X_ts.iloc[tr], y_ts.iloc[tr])
    pred = np.exp(model.predict(X_ts.iloc[te]))
    actual = np.exp(y_ts.iloc[te])
    naive = np.exp(X_ts["lag_12"].iloc[te])                       # seasonal naive: same month last year
    rows.append({"fold": fold + 1, "MAE model": mean_absolute_error(actual, pred),
                 "MAE seasonal naive": mean_absolute_error(actual, naive)})
pd.DataFrame(rows).round(1)

In [ ]:
# Plot the last fold
plt.plot(actual.index, actual, "k.-", label="actual")
plt.plot(actual.index, pred, "o-", label="lag-feature model")
plt.plot(actual.index, naive, "s--", label="seasonal naive")
plt.legend(); plt.title("Forecast for the last 12 months"); plt.show()

## ✏️ Exercises
1. Change `contamination` and `n_neighbors`. How do PR-AUC values move?
2. Replace `LinearRegression` with `HistGradientBoostingRegressor`. Better or worse than the naïve baseline?
3. Remove the `lag_12` feature. What happens to the forecast and why?
4. Load your own monthly series (two columns: date, value) and repeat Part B.